<span style="font-size:2em; font-weight:bold;"> Grenada Rainfall Analysis: Intensity Duration Frequency (IDF) Curve Development - Map of Rainfall extremes across return period (TR)</span>
***

This code was written to produce return period maps using Ordinary Cokriging (OCK) and Ordinary Intrinsic Co-located Cokriging (OICCK) based on the findings from _GRN_IDF_SPATIAL.ipynb_. These return periods are: 5, 10, 25, 50, 75, 100 which fall well within the range of engineering and scientific interest.

# 1.0 Import modules and data

## 1.1 Import modules

In [1]:
# general modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)
from tqdm.auto import tqdm

# spatial statistics
from VarioCorreKrigE.variofit import variofit, crossvariofit
from VarioCorreKrigE.cckrig import align_lmc_experimental_bins, estimate_collocated_correlation, make_covmodel_spec, search_lmc_model_space, ordinary_cokriging, OICCK_MM2

## 1.2 Import data

In [2]:
# gev_tr_summary = pd.read_csv(r'Outputs\gev_tr_summary.csv')
# gpd_tr_summary = pd.read_csv(r'Outputs\gpd_tr_summary.csv')

gev_tr_summary = pd.read_csv(r'Outputs\gev_tr_summary_errsta2.csv')
gpd_tr_summary = pd.read_csv(r'Outputs\gpd_tr_summary_errsta2.csv')

# grenada elevations (secondary values)
elev = pd.read_csv(r'Data/GRENADA_1970DEM_90M_SMOOTH150m_EPSG32620.xyz', header=None)
elev = elev.rename(columns={0:'X_m', 1:'Y_m', 2:'Elevation'})
elev[['X_km','Y_km']] = elev[['X_m','Y_m']]/1000

# z-score standardization
elev['Elev_z'] = (elev['Elevation'] - elev['Elevation'].mean())/(elev['Elevation']).std(ddof = 0)

# 2.0 GEV Return period maps

## 2.1 Mean rainfall extreme

In [3]:
TR_list = sorted(gev_tr_summary.TR.unique())

for TR in tqdm(TR_list, desc='processing TR', total=len(TR_list)):
    values = gev_tr_summary[gev_tr_summary.TR == TR].reset_index(drop=True)
    mu_values = values['TR_mean'].mean()
    sig_values = values['TR_mean'].std(ddof=0)

    # specify fitting parameters
    estimator = "Matheron"
    model = "powered_exponential"
    weight_fn = None
    weight_params = None
    xmax_factor = 2

    # primary data
    h_lag_primary, n_obs_primary, gamma_primary, params_primary, r2_wls_primary, r2_ols_primary = variofit(values = values['TR_z'], coordinates = values[['X_km','Y_km']], distance_type = 'cartesian', max_distance = 15, bin_size = 3, estimator_type= estimator, model_type = model, weight_fn = weight_fn, weight_params = weight_params, xmax_factor=xmax_factor, fix_sill = True , transform="correlation")

    # secondary data
    h_lag_secondary, n_obs_secondary, gamma_secondary, params_secondary, r2_wls_secondary, r2_ols_secondary = variofit(values = elev['Elev_z'], coordinates = elev[['X_km','Y_km']], distance_type = 'cartesian', max_distance = 15, bin_size = 3, estimator_type= estimator, model_type = model, weight_fn = weight_fn, weight_params = weight_params, xmax_factor=xmax_factor, fix_sill = True, fix_nugget = True , transform="correlation")

    # crossvariogram
    h_lag_cross, n_obs_cross, gamma12_cross, params_cross, r2_wls_cross, r2_ols_cross = crossvariofit(values1=values['TR_z'],values2=values['Elev_z'],coordinates=values[['X_km','Y_km']] ,distance_type="cartesian", max_distance=15, bin_size=3, estimator_type=estimator, model_type=model,weight_fn=weight_fn,weight_params=weight_params,fix_nugget=True,fix_sill=True,allow_negative_sill=False, transform="correlation"
    )

    # Rho target
    rho0_target = estimate_collocated_correlation(
    values['TR_z'].to_numpy(),
    values['Elev_z'].to_numpy(),
    method="pearson",
    )

    # LMC
    h_common, rz_common, ry_common, rzy_common, wz_common, wy_common, wzy_common = align_lmc_experimental_bins(
        h_primary=h_lag_primary,
        rho_primary=gamma_primary,
        n_primary=n_obs_primary,
        h_secondary=h_lag_secondary,
        rho_secondary=gamma_secondary,
        n_secondary=n_obs_secondary,
        h_cross=h_lag_cross,
        rho_cross=gamma12_cross,
        n_cross=n_obs_cross,
    )

    search_res = search_lmc_model_space(
        h_common,
        rz_common,
        ry_common,
        rzy_common,
        fixed_structures=None,
        model_family="variogram",
        model_type=model,
        params_primary=params_primary,
        params_secondary=params_secondary,
        params_cross=params_cross,
        n_ranges=9,
        range_spacing="linear",
        range_padding_frac=0.1,
        include_direct_ranges=True,
        include_shape_average=True,
        n_structures=2,
        allow_repeated_kernels=False,
        w_primary=wz_common,
        w_secondary=wy_common,
        w_cross=wzy_common,
        rho0_bounds=[0.2,0.7],
        rho0_target=None,
        cross_misfit_weight_grid=[0.25, 0.5, 1.0, 2.0, 5.0],
        rho0_penalty_weight_grid=[0.0, 1.0, 10.0, 100.0, 1000.0, 1e4],
        normalize_weights=True,
        plot_best=False,
        verbose=False,
        show_progress=False,
    )

    structures = search_res["best"]["fit"]["structures"]

    # Residual correlograms
    # Collocated sample pairs at primary locations
    z_pair = values['TR_z'].to_numpy(dtype=float)
    y_pair = values['Elev_z'].to_numpy(dtype=float)
    xy_pair = values[["X_km", "Y_km"]].to_numpy()

    # Standardize using one consistent reference
    mu_z = z_pair.mean()
    sd_z = z_pair.std(ddof=0)
    mu_y = y_pair.mean()
    sd_y = y_pair.std(ddof=0)

    z_std = (z_pair - mu_z) / sd_z
    y_std = (y_pair - mu_y) / sd_y

    # MM2 residual variable
    r_mm2 = (z_std - rho0_target * y_std) / np.sqrt(1.0 - rho0_target**2)

    # Fit direct correlogram model for the residual variable
    h_lag_resid, n_obs_resid, rho_resid_emp, params_resid, r2_wls_resid, r2_ols_resid = variofit(
        values=r_mm2,
        coordinates=xy_pair,
        distance_type="cartesian",
        max_distance=15,
        bin_size=3,
        estimator_type=estimator,
        model_type=model,
        weight_fn=weight_fn,
        weight_params=weight_params,
        xmax_factor=xmax_factor,
        fix_sill=True,
        fix_nugget=False,
        transform="correlation",
    )

    # make models from correlograms
    primary_model = make_covmodel_spec(
        model_family="variogram",
        model_type="powered_exponential",
        params=params_primary,
    )

    secondary_model = make_covmodel_spec(
        model_family="variogram",
        model_type="powered_exponential",
        params=params_secondary,
    )

    residual_model = make_covmodel_spec(
        model_family="variogram",
        model_type="powered_exponential",
        params=params_resid,
    )

    # kriging
    values = values[~(values.station.isin(["Radix (WTP)", "Bon Accord (WTP)", "Blaize (TANK)", "Grand Etang (LAKE)", "Clozier (Tank)","Plaisance (WTP)", "Munich (WTP)","Peggy's Whim (WTP)"]))].reset_index(drop=True)

    # ICCK MM2
    est_mm2, var_mm2 = OICCK_MM2(
        primary_values=values['TR_z'].to_numpy(),
        primary_coords=values[["X_km", "Y_km"]].to_numpy(),
        secondary_values=values["Elev_z"].to_numpy(),
        secondary_coords=values[["X_km", "Y_km"]].to_numpy(),
        collocated_secondary_values=elev["Elev_z"].to_numpy(),
        targets=elev[["X_km", "Y_km"]].to_numpy(),
        rho0=rho0_target,
        secondary_model=secondary_model,
        residual_model=residual_model,
        secondary_values_for_standardization=values["Elev_z"].to_numpy(),
        distance_type="cartesian",
        rotation_matrix=None,
        standardize=True,
        jitter=1e-10,
        check_positive_definite=True,
        return_weights=False,
        show_progress=False,
    )

    fig, ax = plt.subplots(1, 2, figsize=(12, 6), dpi=200, constrained_layout=True)
    im1 = ax[0].scatter(
        elev["X_m"], elev["Y_m"],
        s=1, c=est_mm2 * sig_values + mu_values,
        cmap="inferno", vmin=np.min(values['TR_mean'].to_numpy()), vmax=np.max(values['TR_mean'].to_numpy())
    )
    im2 = ax[1].scatter(
        elev["X_m"], elev["Y_m"],
        s=1, c=np.sqrt(var_mm2) * sig_values,
        cmap="GnBu"
    )
    ax[0].set_title("Mean")
    ax[1].set_title("Standard Deviation")
    xmin = elev["X_m"].min()
    xmax = elev["X_m"].max()
    ymin = elev["Y_m"].min()
    ymax = elev["Y_m"].max()
    for a in ax:
        a.set_aspect("equal")
        a.set_xlim(xmin, xmax)
        a.set_ylim(ymin, ymax)
        a.set_xlabel("Easting (m)")
        a.set_ylabel("Northing (m)")
    cbar1 = fig.colorbar(im1, ax=ax[0], orientation="vertical",fraction=0.03, pad=0.02)
    cbar1.set_label(r"$24-Hour Rainfall$ (mm/day)")
    cbar2 = fig.colorbar(im2, ax=ax[1], orientation="vertical",fraction=0.03, pad=0.02)
    cbar2.set_label(r"$24-Hour Rainfall$ (mm/day)")
    plt.close('all')

    # OCK
    est_ock, var_ock = ordinary_cokriging(
        primary_values=values['TR_z'].to_numpy(),
        primary_coords=values[["X_km", "Y_km"]].to_numpy(),
        secondary_values=elev["Elev_z"].to_numpy(),
        secondary_coords=elev[["X_km", "Y_km"]].to_numpy(),
        targets=elev[["X_km", "Y_km"]].to_numpy(),
        covariance_mode="lmc",
        structures=structures,
        distance_type="cartesian",
        rotation_matrix=None,
        standardize=True,
        jitter=1e-10,
        check_positive_definite=True,
        return_weights=False,
        max_neighbors_secondary=256,
        max_neighbors_primary=256,
        show_progress=False,
    )

    fig, ax = plt.subplots(1, 2, figsize=(12, 6), dpi=200, constrained_layout=True)
    im1 = ax[0].scatter(
        elev["X_m"], elev["Y_m"],
        s=1, c=est_ock * sig_values + mu_values,
        cmap="inferno", vmin=np.min(values['TR_mean'].to_numpy()), vmax=np.max(values['TR_mean'].to_numpy())
    )
    im2 = ax[1].scatter(
        elev["X_m"], elev["Y_m"],
        s=1, c=np.sqrt(var_ock) * sig_values,
        cmap="GnBu"
    )
    ax[0].set_title("Mean")
    ax[1].set_title("Standard Deviation")
    xmin = elev["X_m"].min()
    xmax = elev["X_m"].max()
    ymin = elev["Y_m"].min()
    ymax = elev["Y_m"].max()
    for a in ax:
        a.set_aspect("equal")
        a.set_xlim(xmin, xmax)
        a.set_ylim(ymin, ymax)
        a.set_xlabel("Easting (m)")
        a.set_ylabel("Northing (m)")
    cbar1 = fig.colorbar(im1, ax=ax[0], orientation="vertical",fraction=0.03, pad=0.02)
    cbar1.set_label(r"$24-Hour Rainfall$ (mm/day)")
    cbar2 = fig.colorbar(im2, ax=ax[1], orientation="vertical",fraction=0.03, pad=0.02)
    cbar2.set_label(r"$24-Hour Rainfall$ (mm/day)")
    plt.close('all')

    # store values
    elev[f'ock_{TR}_mean'] = est_ock * sig_values + mu_values
    elev[f'ock_{TR}_mean_std'] = np.sqrt(var_ock) * sig_values
    elev[f'oicckm2_{TR}_mean'] = est_mm2 * sig_values + mu_values
    elev[f'oicckm2_{TR}_mean_std'] = np.sqrt(var_mm2) * sig_values

processing TR:   0%|          | 0/6 [00:00<?, ?it/s]

## 2.2 Standard deviation rainfall extreme

In [4]:
TR_list = sorted(gev_tr_summary.TR.unique())

for TR in tqdm(TR_list, desc='processing TR', total=len(TR_list)):
    values = gev_tr_summary[gev_tr_summary.TR == TR].reset_index(drop=True)
    mu_values = values['TR_std'].mean()
    sig_values = values['TR_std'].std(ddof=0)

    # specify fitting parameters
    estimator = "Matheron"
    model = "powered_exponential"
    weight_fn = None
    weight_params = None
    xmax_factor = 2

    # primary data
    h_lag_primary, n_obs_primary, gamma_primary, params_primary, r2_wls_primary, r2_ols_primary = variofit(values = values['TR_std_z'], coordinates = values[['X_km','Y_km']], distance_type = 'cartesian', max_distance = 15, bin_size = 3, estimator_type= estimator, model_type = model, weight_fn = weight_fn, weight_params = weight_params, xmax_factor=xmax_factor, fix_sill = True , transform="correlation")

    # secondary data
    h_lag_secondary, n_obs_secondary, gamma_secondary, params_secondary, r2_wls_secondary, r2_ols_secondary = variofit(values = elev['Elev_z'], coordinates = elev[['X_km','Y_km']], distance_type = 'cartesian', max_distance = 15, bin_size = 3, estimator_type= estimator, model_type = model, weight_fn = weight_fn, weight_params = weight_params, xmax_factor=xmax_factor, fix_sill = True, fix_nugget = True , transform="correlation")

    # crossvariogram
    h_lag_cross, n_obs_cross, gamma12_cross, params_cross, r2_wls_cross, r2_ols_cross = crossvariofit(values1=values['TR_std_z'],values2=values['Elev_z'],coordinates=values[['X_km','Y_km']] ,distance_type="cartesian", max_distance=15, bin_size=3, estimator_type=estimator, model_type=model,weight_fn=weight_fn,weight_params=weight_params,fix_nugget=True,fix_sill=True,allow_negative_sill=False, transform="correlation"
    )

    # Rho target
    rho0_target = estimate_collocated_correlation(
    values['TR_std_z'].to_numpy(),
    values['Elev_z'].to_numpy(),
    method="pearson",
    )

    # LMC
    h_common, rz_common, ry_common, rzy_common, wz_common, wy_common, wzy_common = align_lmc_experimental_bins(
        h_primary=h_lag_primary,
        rho_primary=gamma_primary,
        n_primary=n_obs_primary,
        h_secondary=h_lag_secondary,
        rho_secondary=gamma_secondary,
        n_secondary=n_obs_secondary,
        h_cross=h_lag_cross,
        rho_cross=gamma12_cross,
        n_cross=n_obs_cross,
    )

    search_res = search_lmc_model_space(
        h_common,
        rz_common,
        ry_common,
        rzy_common,
        fixed_structures=None,
        model_family="variogram",
        model_type=model,
        params_primary=params_primary,
        params_secondary=params_secondary,
        params_cross=params_cross,
        n_ranges=9,
        range_spacing="linear",
        range_padding_frac=0.1,
        include_direct_ranges=True,
        include_shape_average=True,
        n_structures=2,
        allow_repeated_kernels=False,
        w_primary=wz_common,
        w_secondary=wy_common,
        w_cross=wzy_common,
        rho0_bounds=[0.2,0.7],
        rho0_target=None,
        cross_misfit_weight_grid=[0.25, 0.5, 1.0, 2.0, 5.0],
        rho0_penalty_weight_grid=[0.0, 1.0, 10.0, 100.0, 1000.0, 1e4],
        normalize_weights=True,
        plot_best=False,
        verbose=False,
        show_progress=False,
    )

    structures = search_res["best"]["fit"]["structures"]

    # Residual correlograms
    # Collocated sample pairs at primary locations
    z_pair = values['TR_std_z'].to_numpy(dtype=float)
    y_pair = values['Elev_z'].to_numpy(dtype=float)
    xy_pair = values[["X_km", "Y_km"]].to_numpy()

    # Standardize using one consistent reference
    mu_z = z_pair.mean()
    sd_z = z_pair.std(ddof=0)
    mu_y = y_pair.mean()
    sd_y = y_pair.std(ddof=0)

    z_std = (z_pair - mu_z) / sd_z
    y_std = (y_pair - mu_y) / sd_y

    # MM2 residual variable
    r_mm2 = (z_std - rho0_target * y_std) / np.sqrt(1.0 - rho0_target**2)

    # Fit direct correlogram model for the residual variable
    h_lag_resid, n_obs_resid, rho_resid_emp, params_resid, r2_wls_resid, r2_ols_resid = variofit(
        values=r_mm2,
        coordinates=xy_pair,
        distance_type="cartesian",
        max_distance=15,
        bin_size=3,
        estimator_type=estimator,
        model_type=model,
        weight_fn=weight_fn,
        weight_params=weight_params,
        xmax_factor=xmax_factor,
        fix_sill=True,
        fix_nugget=False,
        transform="correlation",
    )

    # make models from correlograms
    primary_model = make_covmodel_spec(
        model_family="variogram",
        model_type="powered_exponential",
        params=params_primary,
    )

    secondary_model = make_covmodel_spec(
        model_family="variogram",
        model_type="powered_exponential",
        params=params_secondary,
    )

    residual_model = make_covmodel_spec(
        model_family="variogram",
        model_type="powered_exponential",
        params=params_resid,
    )

    # kriging
    values = values[~(values.station.isin(["Radix (WTP)", "Bon Accord (WTP)", "Blaize (TANK)", "Grand Etang (LAKE)", "Clozier (Tank)","Plaisance (WTP)", "Munich (WTP)","Peggy's Whim (WTP)"]))].reset_index(drop=True)

    # ICCK MM2
    est_mm2, var_mm2 = OICCK_MM2(
        primary_values=values['TR_std_z'].to_numpy(),
        primary_coords=values[["X_km", "Y_km"]].to_numpy(),
        secondary_values=values["Elev_z"].to_numpy(),
        secondary_coords=values[["X_km", "Y_km"]].to_numpy(),
        collocated_secondary_values=elev["Elev_z"].to_numpy(),
        targets=elev[["X_km", "Y_km"]].to_numpy(),
        rho0=rho0_target,
        secondary_model=secondary_model,
        residual_model=residual_model,
        secondary_values_for_standardization=values["Elev_z"].to_numpy(),
        distance_type="cartesian",
        rotation_matrix=None,
        standardize=True,
        jitter=1e-10,
        check_positive_definite=True,
        return_weights=False,
        show_progress=False,
    )

    fig, ax = plt.subplots(1, 2, figsize=(12, 6), dpi=200, constrained_layout=True)
    im1 = ax[0].scatter(
        elev["X_m"], elev["Y_m"],
        s=1, c=est_mm2 * sig_values + mu_values,
        cmap="inferno", vmin=np.min(values['TR_std'].to_numpy()), vmax=np.max(values['TR_std'].to_numpy())
    )
    im2 = ax[1].scatter(
        elev["X_m"], elev["Y_m"],
        s=1, c=np.sqrt(var_mm2) * sig_values,
        cmap="GnBu"
    )
    ax[0].set_title("Mean")
    ax[1].set_title("Standard Deviation")
    xmin = elev["X_m"].min()
    xmax = elev["X_m"].max()
    ymin = elev["Y_m"].min()
    ymax = elev["Y_m"].max()
    for a in ax:
        a.set_aspect("equal")
        a.set_xlim(xmin, xmax)
        a.set_ylim(ymin, ymax)
        a.set_xlabel("Easting (m)")
        a.set_ylabel("Northing (m)")
    cbar1 = fig.colorbar(im1, ax=ax[0], orientation="vertical",fraction=0.03, pad=0.02)
    cbar1.set_label(r"$24-Hour Rainfall$ (mm/day)")
    cbar2 = fig.colorbar(im2, ax=ax[1], orientation="vertical",fraction=0.03, pad=0.02)
    cbar2.set_label(r"$24-Hour Rainfall$ (mm/day)")
    plt.close('all')

    # OCK
    est_ock, var_ock = ordinary_cokriging(
        primary_values=values['TR_std_z'].to_numpy(),
        primary_coords=values[["X_km", "Y_km"]].to_numpy(),
        secondary_values=elev["Elev_z"].to_numpy(),
        secondary_coords=elev[["X_km", "Y_km"]].to_numpy(),
        targets=elev[["X_km", "Y_km"]].to_numpy(),
        covariance_mode="lmc",
        structures=structures,
        distance_type="cartesian",
        rotation_matrix=None,
        standardize=True,
        jitter=1e-10,
        check_positive_definite=True,
        return_weights=False,
        max_neighbors_secondary=256,
        max_neighbors_primary=256,
        show_progress=False,
    )

    fig, ax = plt.subplots(1, 2, figsize=(12, 6), dpi=200, constrained_layout=True)
    im1 = ax[0].scatter(
        elev["X_m"], elev["Y_m"],
        s=1, c=est_ock * sig_values + mu_values,
        cmap="inferno", vmin=np.min(values['TR_std'].to_numpy()), vmax=np.max(values['TR_std'].to_numpy())
    )
    im2 = ax[1].scatter(
        elev["X_m"], elev["Y_m"],
        s=1, c=np.sqrt(var_ock) * sig_values,
        cmap="GnBu"
    )
    ax[0].set_title("Mean")
    ax[1].set_title("Standard Deviation")
    xmin = elev["X_m"].min()
    xmax = elev["X_m"].max()
    ymin = elev["Y_m"].min()
    ymax = elev["Y_m"].max()
    for a in ax:
        a.set_aspect("equal")
        a.set_xlim(xmin, xmax)
        a.set_ylim(ymin, ymax)
        a.set_xlabel("Easting (m)")
        a.set_ylabel("Northing (m)")
    cbar1 = fig.colorbar(im1, ax=ax[0], orientation="vertical",fraction=0.03, pad=0.02)
    cbar1.set_label(r"$24-Hour Rainfall$ (mm/day)")
    cbar2 = fig.colorbar(im2, ax=ax[1], orientation="vertical",fraction=0.03, pad=0.02)
    cbar2.set_label(r"$24-Hour Rainfall$ (mm/day)")
    plt.close('all')

    # store values
    elev[f'ock_{TR}_std'] = est_ock * sig_values + mu_values
    elev[f'ock_{TR}_std_std'] = np.sqrt(var_ock) * sig_values
    elev[f'oicckm2_{TR}_std'] = est_mm2 * sig_values + mu_values
    elev[f'oicckm2_{TR}_std_std'] = np.sqrt(var_mm2) * sig_values

processing TR:   0%|          | 0/6 [00:00<?, ?it/s]

## 2.3 Export data

In [5]:
elev_export = elev.drop(columns={'X_km', 'Y_km', 'Elev_z'})
pred_cols = [c for c in elev_export.columns if 'ock_' in c or 'oicckm2_' in c]
elev_export[pred_cols] = elev_export[pred_cols].round(3)
elev_export.to_csv(r'Outputs\gev_tr_map_values_errsta2.csv', index=False)

# 3.0 GPD Return period maps

## 3.1 Mean rainfall extreme

In [6]:
TR_list = sorted(gpd_tr_summary.TR.unique())

for TR in tqdm(TR_list, desc='processing TR', total=len(TR_list)):
    values = gpd_tr_summary[gpd_tr_summary.TR == TR].reset_index(drop=True)
    mu_values = values['TR_mean'].mean()
    sig_values = values['TR_mean'].std(ddof=0)

    # specify fitting parameters
    estimator = "Matheron"
    model = "powered_exponential"
    weight_fn = None
    weight_params = None
    xmax_factor = 2

    # primary data
    h_lag_primary, n_obs_primary, gamma_primary, params_primary, r2_wls_primary, r2_ols_primary = variofit(values = values['TR_z'], coordinates = values[['X_km','Y_km']], distance_type = 'cartesian', max_distance = 15, bin_size = 3, estimator_type= estimator, model_type = model, weight_fn = weight_fn, weight_params = weight_params, xmax_factor=xmax_factor, fix_sill = True , transform="correlation")

    # secondary data
    h_lag_secondary, n_obs_secondary, gamma_secondary, params_secondary, r2_wls_secondary, r2_ols_secondary = variofit(values = elev['Elev_z'], coordinates = elev[['X_km','Y_km']], distance_type = 'cartesian', max_distance = 15, bin_size = 3, estimator_type= estimator, model_type = model, weight_fn = weight_fn, weight_params = weight_params, xmax_factor=xmax_factor, fix_sill = True, fix_nugget = True , transform="correlation")

    # crossvariogram
    h_lag_cross, n_obs_cross, gamma12_cross, params_cross, r2_wls_cross, r2_ols_cross = crossvariofit(values1=values['TR_z'],values2=values['Elev_z'],coordinates=values[['X_km','Y_km']] ,distance_type="cartesian", max_distance=15, bin_size=3, estimator_type=estimator, model_type=model,weight_fn=weight_fn,weight_params=weight_params,fix_nugget=True,fix_sill=True,allow_negative_sill=False, transform="correlation"
    )

    # Rho target
    rho0_target = estimate_collocated_correlation(
    values['TR_z'].to_numpy(),
    values['Elev_z'].to_numpy(),
    method="pearson",
    )

    # LMC
    h_common, rz_common, ry_common, rzy_common, wz_common, wy_common, wzy_common = align_lmc_experimental_bins(
        h_primary=h_lag_primary,
        rho_primary=gamma_primary,
        n_primary=n_obs_primary,
        h_secondary=h_lag_secondary,
        rho_secondary=gamma_secondary,
        n_secondary=n_obs_secondary,
        h_cross=h_lag_cross,
        rho_cross=gamma12_cross,
        n_cross=n_obs_cross,
    )

    search_res = search_lmc_model_space(
        h_common,
        rz_common,
        ry_common,
        rzy_common,
        fixed_structures=None,
        model_family="variogram",
        model_type=model,
        params_primary=params_primary,
        params_secondary=params_secondary,
        params_cross=params_cross,
        n_ranges=9,
        range_spacing="linear",
        range_padding_frac=0.1,
        include_direct_ranges=True,
        include_shape_average=True,
        n_structures=2,
        allow_repeated_kernels=False,
        w_primary=wz_common,
        w_secondary=wy_common,
        w_cross=wzy_common,
        rho0_bounds=[0.2,0.7],
        rho0_target=None,
        cross_misfit_weight_grid=[0.25, 0.5, 1.0, 2.0, 5.0],
        rho0_penalty_weight_grid=[0.0, 1.0, 10.0, 100.0, 1000.0, 1e4],
        normalize_weights=True,
        plot_best=False,
        verbose=False,
        show_progress=False,
    )

    structures = search_res["best"]["fit"]["structures"]

    # Residual correlograms
    # Collocated sample pairs at primary locations
    z_pair = values['TR_z'].to_numpy(dtype=float)
    y_pair = values['Elev_z'].to_numpy(dtype=float)
    xy_pair = values[["X_km", "Y_km"]].to_numpy()

    # Standardize using one consistent reference
    mu_z = z_pair.mean()
    sd_z = z_pair.std(ddof=0)
    mu_y = y_pair.mean()
    sd_y = y_pair.std(ddof=0)

    z_std = (z_pair - mu_z) / sd_z
    y_std = (y_pair - mu_y) / sd_y

    # MM2 residual variable
    r_mm2 = (z_std - rho0_target * y_std) / np.sqrt(1.0 - rho0_target**2)

    # Fit direct correlogram model for the residual variable
    h_lag_resid, n_obs_resid, rho_resid_emp, params_resid, r2_wls_resid, r2_ols_resid = variofit(
        values=r_mm2,
        coordinates=xy_pair,
        distance_type="cartesian",
        max_distance=15,
        bin_size=3,
        estimator_type=estimator,
        model_type=model,
        weight_fn=weight_fn,
        weight_params=weight_params,
        xmax_factor=xmax_factor,
        fix_sill=True,
        fix_nugget=False,
        transform="correlation",
    )

    # make models from correlograms
    primary_model = make_covmodel_spec(
        model_family="variogram",
        model_type="powered_exponential",
        params=params_primary,
    )

    secondary_model = make_covmodel_spec(
        model_family="variogram",
        model_type="powered_exponential",
        params=params_secondary,
    )

    residual_model = make_covmodel_spec(
        model_family="variogram",
        model_type="powered_exponential",
        params=params_resid,
    )

    # kriging
    values = values[~(values.station.isin(["Radix (WTP)", "Bon Accord (WTP)", "Blaize (TANK)", "Grand Etang (LAKE)", "Clozier (Tank)","Plaisance (WTP)", "Munich (WTP)","Peggy's Whim (WTP)"]))].reset_index(drop=True)

    # ICCK MM2
    est_mm2, var_mm2 = OICCK_MM2(
        primary_values=values['TR_z'].to_numpy(),
        primary_coords=values[["X_km", "Y_km"]].to_numpy(),
        secondary_values=values["Elev_z"].to_numpy(),
        secondary_coords=values[["X_km", "Y_km"]].to_numpy(),
        collocated_secondary_values=elev["Elev_z"].to_numpy(),
        targets=elev[["X_km", "Y_km"]].to_numpy(),
        rho0=rho0_target,
        secondary_model=secondary_model,
        residual_model=residual_model,
        secondary_values_for_standardization=values["Elev_z"].to_numpy(),
        distance_type="cartesian",
        rotation_matrix=None,
        standardize=True,
        jitter=1e-10,
        check_positive_definite=True,
        return_weights=False,
        show_progress=False,
    )

    fig, ax = plt.subplots(1, 2, figsize=(12, 6), dpi=200, constrained_layout=True)
    im1 = ax[0].scatter(
        elev["X_m"], elev["Y_m"],
        s=1, c=est_mm2 * sig_values + mu_values,
        cmap="inferno", vmin=np.min(values['TR_mean'].to_numpy()), vmax=np.max(values['TR_mean'].to_numpy())
    )
    im2 = ax[1].scatter(
        elev["X_m"], elev["Y_m"],
        s=1, c=np.sqrt(var_mm2) * sig_values,
        cmap="GnBu"
    )
    ax[0].set_title("Mean")
    ax[1].set_title("Standard Deviation")
    xmin = elev["X_m"].min()
    xmax = elev["X_m"].max()
    ymin = elev["Y_m"].min()
    ymax = elev["Y_m"].max()
    for a in ax:
        a.set_aspect("equal")
        a.set_xlim(xmin, xmax)
        a.set_ylim(ymin, ymax)
        a.set_xlabel("Easting (m)")
        a.set_ylabel("Northing (m)")
    cbar1 = fig.colorbar(im1, ax=ax[0], orientation="vertical",fraction=0.03, pad=0.02)
    cbar1.set_label(r"$24-Hour Rainfall$ (mm/day)")
    cbar2 = fig.colorbar(im2, ax=ax[1], orientation="vertical",fraction=0.03, pad=0.02)
    cbar2.set_label(r"$24-Hour Rainfall$ (mm/day)")
    plt.close('all')

    # OCK
    est_ock, var_ock = ordinary_cokriging(
        primary_values=values['TR_z'].to_numpy(),
        primary_coords=values[["X_km", "Y_km"]].to_numpy(),
        secondary_values=elev["Elev_z"].to_numpy(),
        secondary_coords=elev[["X_km", "Y_km"]].to_numpy(),
        targets=elev[["X_km", "Y_km"]].to_numpy(),
        covariance_mode="lmc",
        structures=structures,
        distance_type="cartesian",
        rotation_matrix=None,
        standardize=True,
        jitter=1e-10,
        check_positive_definite=True,
        return_weights=False,
        max_neighbors_secondary=256,
        max_neighbors_primary=256,
        show_progress=False,
    )

    fig, ax = plt.subplots(1, 2, figsize=(12, 6), dpi=200, constrained_layout=True)
    im1 = ax[0].scatter(
        elev["X_m"], elev["Y_m"],
        s=1, c=est_ock * sig_values + mu_values,
        cmap="inferno", vmin=np.min(values['TR_mean'].to_numpy()), vmax=np.max(values['TR_mean'].to_numpy())
    )
    im2 = ax[1].scatter(
        elev["X_m"], elev["Y_m"],
        s=1, c=np.sqrt(var_ock) * sig_values,
        cmap="GnBu"
    )
    ax[0].set_title("Mean")
    ax[1].set_title("Standard Deviation")
    xmin = elev["X_m"].min()
    xmax = elev["X_m"].max()
    ymin = elev["Y_m"].min()
    ymax = elev["Y_m"].max()
    for a in ax:
        a.set_aspect("equal")
        a.set_xlim(xmin, xmax)
        a.set_ylim(ymin, ymax)
        a.set_xlabel("Easting (m)")
        a.set_ylabel("Northing (m)")
    cbar1 = fig.colorbar(im1, ax=ax[0], orientation="vertical",fraction=0.03, pad=0.02)
    cbar1.set_label(r"$24-Hour Rainfall$ (mm/day)")
    cbar2 = fig.colorbar(im2, ax=ax[1], orientation="vertical",fraction=0.03, pad=0.02)
    cbar2.set_label(r"$24-Hour Rainfall$ (mm/day)")
    plt.close('all')

    # store values
    elev[f'ock_{TR}_mean'] = est_ock * sig_values + mu_values
    elev[f'ock_{TR}_mean_std'] = np.sqrt(var_ock) * sig_values
    elev[f'oicckm2_{TR}_mean'] = est_mm2 * sig_values + mu_values
    elev[f'oicckm2_{TR}_mean_std'] = np.sqrt(var_mm2) * sig_values

processing TR:   0%|          | 0/6 [00:00<?, ?it/s]

## 3.2 Standard deviation rainfall extreme

In [7]:
TR_list = sorted(gpd_tr_summary.TR.unique())

for TR in tqdm(TR_list, desc='processing TR', total=len(TR_list)):
    values = gpd_tr_summary[gpd_tr_summary.TR == TR].reset_index(drop=True)
    mu_values = values['TR_std'].mean()
    sig_values = values['TR_std'].std(ddof=0)

    # specify fitting parameters
    estimator = "Matheron"
    model = "powered_exponential"
    weight_fn = None
    weight_params = None
    xmax_factor = 2

    # primary data
    h_lag_primary, n_obs_primary, gamma_primary, params_primary, r2_wls_primary, r2_ols_primary = variofit(values = values['TR_std_z'], coordinates = values[['X_km','Y_km']], distance_type = 'cartesian', max_distance = 15, bin_size = 3, estimator_type= estimator, model_type = model, weight_fn = weight_fn, weight_params = weight_params, xmax_factor=xmax_factor, fix_sill = True , transform="correlation")

    # secondary data
    h_lag_secondary, n_obs_secondary, gamma_secondary, params_secondary, r2_wls_secondary, r2_ols_secondary = variofit(values = elev['Elev_z'], coordinates = elev[['X_km','Y_km']], distance_type = 'cartesian', max_distance = 15, bin_size = 3, estimator_type= estimator, model_type = model, weight_fn = weight_fn, weight_params = weight_params, xmax_factor=xmax_factor, fix_sill = True, fix_nugget = True , transform="correlation")

    # crossvariogram
    h_lag_cross, n_obs_cross, gamma12_cross, params_cross, r2_wls_cross, r2_ols_cross = crossvariofit(values1=values['TR_std_z'],values2=values['Elev_z'],coordinates=values[['X_km','Y_km']] ,distance_type="cartesian", max_distance=15, bin_size=3, estimator_type=estimator, model_type=model,weight_fn=weight_fn,weight_params=weight_params,fix_nugget=True,fix_sill=True,allow_negative_sill=False, transform="correlation"
    )

    # Rho target
    rho0_target = estimate_collocated_correlation(
    values['TR_std_z'].to_numpy(),
    values['Elev_z'].to_numpy(),
    method="pearson",
    )

    # LMC
    h_common, rz_common, ry_common, rzy_common, wz_common, wy_common, wzy_common = align_lmc_experimental_bins(
        h_primary=h_lag_primary,
        rho_primary=gamma_primary,
        n_primary=n_obs_primary,
        h_secondary=h_lag_secondary,
        rho_secondary=gamma_secondary,
        n_secondary=n_obs_secondary,
        h_cross=h_lag_cross,
        rho_cross=gamma12_cross,
        n_cross=n_obs_cross,
    )

    search_res = search_lmc_model_space(
        h_common,
        rz_common,
        ry_common,
        rzy_common,
        fixed_structures=None,
        model_family="variogram",
        model_type=model,
        params_primary=params_primary,
        params_secondary=params_secondary,
        params_cross=params_cross,
        n_ranges=9,
        range_spacing="linear",
        range_padding_frac=0.1,
        include_direct_ranges=True,
        include_shape_average=True,
        n_structures=2,
        allow_repeated_kernels=False,
        w_primary=wz_common,
        w_secondary=wy_common,
        w_cross=wzy_common,
        rho0_bounds=[0.2,0.7],
        rho0_target=None,
        cross_misfit_weight_grid=[0.25, 0.5, 1.0, 2.0, 5.0],
        rho0_penalty_weight_grid=[0.0, 1.0, 10.0, 100.0, 1000.0, 1e4],
        normalize_weights=True,
        plot_best=False,
        verbose=False,
        show_progress=False,
    )

    structures = search_res["best"]["fit"]["structures"]

    # Residual correlograms
    # Collocated sample pairs at primary locations
    z_pair = values['TR_std_z'].to_numpy(dtype=float)
    y_pair = values['Elev_z'].to_numpy(dtype=float)
    xy_pair = values[["X_km", "Y_km"]].to_numpy()

    # Standardize using one consistent reference
    mu_z = z_pair.mean()
    sd_z = z_pair.std(ddof=0)
    mu_y = y_pair.mean()
    sd_y = y_pair.std(ddof=0)

    z_std = (z_pair - mu_z) / sd_z
    y_std = (y_pair - mu_y) / sd_y

    # MM2 residual variable
    r_mm2 = (z_std - rho0_target * y_std) / np.sqrt(1.0 - rho0_target**2)

    # Fit direct correlogram model for the residual variable
    h_lag_resid, n_obs_resid, rho_resid_emp, params_resid, r2_wls_resid, r2_ols_resid = variofit(
        values=r_mm2,
        coordinates=xy_pair,
        distance_type="cartesian",
        max_distance=15,
        bin_size=3,
        estimator_type=estimator,
        model_type=model,
        weight_fn=weight_fn,
        weight_params=weight_params,
        xmax_factor=xmax_factor,
        fix_sill=True,
        fix_nugget=False,
        transform="correlation",
    )

    # make models from correlograms
    primary_model = make_covmodel_spec(
        model_family="variogram",
        model_type="powered_exponential",
        params=params_primary,
    )

    secondary_model = make_covmodel_spec(
        model_family="variogram",
        model_type="powered_exponential",
        params=params_secondary,
    )

    residual_model = make_covmodel_spec(
        model_family="variogram",
        model_type="powered_exponential",
        params=params_resid,
    )

    # kriging
    values = values[~(values.station.isin(["Radix (WTP)", "Bon Accord (WTP)", "Blaize (TANK)", "Grand Etang (LAKE)", "Clozier (Tank)","Plaisance (WTP)", "Munich (WTP)","Peggy's Whim (WTP)"]))].reset_index(drop=True)

    # ICCK MM2
    est_mm2, var_mm2 = OICCK_MM2(
        primary_values=values['TR_std_z'].to_numpy(),
        primary_coords=values[["X_km", "Y_km"]].to_numpy(),
        secondary_values=values["Elev_z"].to_numpy(),
        secondary_coords=values[["X_km", "Y_km"]].to_numpy(),
        collocated_secondary_values=elev["Elev_z"].to_numpy(),
        targets=elev[["X_km", "Y_km"]].to_numpy(),
        rho0=rho0_target,
        secondary_model=secondary_model,
        residual_model=residual_model,
        secondary_values_for_standardization=values["Elev_z"].to_numpy(),
        distance_type="cartesian",
        rotation_matrix=None,
        standardize=True,
        jitter=1e-10,
        check_positive_definite=True,
        return_weights=False,
        show_progress=False,
    )

    fig, ax = plt.subplots(1, 2, figsize=(12, 6), dpi=200, constrained_layout=True)
    im1 = ax[0].scatter(
        elev["X_m"], elev["Y_m"],
        s=1, c=est_mm2 * sig_values + mu_values,
        cmap="inferno", vmin=np.min(values['TR_std'].to_numpy()), vmax=np.max(values['TR_std'].to_numpy())
    )
    im2 = ax[1].scatter(
        elev["X_m"], elev["Y_m"],
        s=1, c=np.sqrt(var_mm2) * sig_values,
        cmap="GnBu"
    )
    ax[0].set_title("Mean")
    ax[1].set_title("Standard Deviation")
    xmin = elev["X_m"].min()
    xmax = elev["X_m"].max()
    ymin = elev["Y_m"].min()
    ymax = elev["Y_m"].max()
    for a in ax:
        a.set_aspect("equal")
        a.set_xlim(xmin, xmax)
        a.set_ylim(ymin, ymax)
        a.set_xlabel("Easting (m)")
        a.set_ylabel("Northing (m)")
    cbar1 = fig.colorbar(im1, ax=ax[0], orientation="vertical",fraction=0.03, pad=0.02)
    cbar1.set_label(r"$24-Hour Rainfall$ (mm/day)")
    cbar2 = fig.colorbar(im2, ax=ax[1], orientation="vertical",fraction=0.03, pad=0.02)
    cbar2.set_label(r"$24-Hour Rainfall$ (mm/day)")
    plt.close('all')

    # OCK
    est_ock, var_ock = ordinary_cokriging(
        primary_values=values['TR_std_z'].to_numpy(),
        primary_coords=values[["X_km", "Y_km"]].to_numpy(),
        secondary_values=elev["Elev_z"].to_numpy(),
        secondary_coords=elev[["X_km", "Y_km"]].to_numpy(),
        targets=elev[["X_km", "Y_km"]].to_numpy(),
        covariance_mode="lmc",
        structures=structures,
        distance_type="cartesian",
        rotation_matrix=None,
        standardize=True,
        jitter=1e-10,
        check_positive_definite=True,
        return_weights=False,
        max_neighbors_secondary=256,
        max_neighbors_primary=256,
        show_progress=False,
    )

    fig, ax = plt.subplots(1, 2, figsize=(12, 6), dpi=200, constrained_layout=True)
    im1 = ax[0].scatter(
        elev["X_m"], elev["Y_m"],
        s=1, c=est_ock * sig_values + mu_values,
        cmap="inferno", vmin=np.min(values['TR_std'].to_numpy()), vmax=np.max(values['TR_std'].to_numpy())
    )
    im2 = ax[1].scatter(
        elev["X_m"], elev["Y_m"],
        s=1, c=np.sqrt(var_ock) * sig_values,
        cmap="GnBu"
    )
    ax[0].set_title("Mean")
    ax[1].set_title("Standard Deviation")
    xmin = elev["X_m"].min()
    xmax = elev["X_m"].max()
    ymin = elev["Y_m"].min()
    ymax = elev["Y_m"].max()
    for a in ax:
        a.set_aspect("equal")
        a.set_xlim(xmin, xmax)
        a.set_ylim(ymin, ymax)
        a.set_xlabel("Easting (m)")
        a.set_ylabel("Northing (m)")
    cbar1 = fig.colorbar(im1, ax=ax[0], orientation="vertical",fraction=0.03, pad=0.02)
    cbar1.set_label(r"$24-Hour Rainfall$ (mm/day)")
    cbar2 = fig.colorbar(im2, ax=ax[1], orientation="vertical",fraction=0.03, pad=0.02)
    cbar2.set_label(r"$24-Hour Rainfall$ (mm/day)")
    plt.close('all')

    # store values
    elev[f'ock_{TR}_std'] = est_ock * sig_values + mu_values
    elev[f'ock_{TR}_std_std'] = np.sqrt(var_ock) * sig_values
    elev[f'oicckm2_{TR}_std'] = est_mm2 * sig_values + mu_values
    elev[f'oicckm2_{TR}_std_std'] = np.sqrt(var_mm2) * sig_values

processing TR:   0%|          | 0/6 [00:00<?, ?it/s]

## 3.3 Export data

In [8]:
elev_export = elev.drop(columns={'X_km', 'Y_km', 'Elev_z'})
pred_cols = [c for c in elev_export.columns if 'ock_' in c or 'oicckm2_' in c]
elev_export[pred_cols] = elev_export[pred_cols].round(3)
elev_export.to_csv(r'Outputs\gpd_tr_map_values_errsta2.csv', index=False)